In [81]:
import sys
sys.path.insert(0, '/Users/miranda/code/mit/summer26/ahri/SIRL/miranda/src')
from utils import *
from tversky_sirl_2 import *

FPE_SPLIT_SEED = 42
SIRL_SEED = 0
n_queries = 1000

print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
anchors, positives, negatives = load_data(n_queries)
print(anchors.shape)

print(f"  -- seed {SIRL_SEED} --")
set_all_seeds(SIRL_SEED)

model, _ = train_tversky_sirl(
    anchors, positives, negatives,
    fbank_size=4,
    use_symmetric_loss=True,
)


=== TRAINING SIRL FOR 1000 QUERIES ===
(1000, 21, 27)
  -- seed 0 --
Epoch    0 | loss=4267075840.0000 | triplet_acc=0.463 | lr=0.00400
Epoch  100 | loss=75327760.0000 | triplet_acc=0.462 | lr=0.00400
Epoch  200 | loss=28519340.0000 | triplet_acc=0.462 | lr=0.00399
Epoch  300 | loss=10756356.0000 | triplet_acc=0.463 | lr=0.00399
Epoch  400 | loss=5427636.0000 | triplet_acc=0.463 | lr=0.00398
Epoch  500 | loss=3423467.7500 | triplet_acc=0.463 | lr=0.00398
Epoch  600 | loss=2032623.5000 | triplet_acc=0.463 | lr=0.00398
Epoch  700 | loss=1214578.6250 | triplet_acc=0.465 | lr=0.00397
Epoch  800 | loss=732416.7500 | triplet_acc=0.464 | lr=0.00397
Epoch  900 | loss=626961.5625 | triplet_acc=0.464 | lr=0.00396
Epoch 1000 | loss=359995.1875 | triplet_acc=0.464 | lr=0.00396
Epoch 1100 | loss=221214.5312 | triplet_acc=0.467 | lr=0.00396
Epoch 1200 | loss=177173.2812 | triplet_acc=0.467 | lr=0.00395
Epoch 1300 | loss=122563.9922 | triplet_acc=0.467 | lr=0.00395
Epoch 1400 | loss=77094.0156 | tri

In [82]:
all_trajs = load_all_trajs()
all_features = get_features(all_trajs, all_trajs)

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

In [83]:
normalized_all_trajs = tx(all_trajs)
all_embeds = model(torch.tensor(normalized_all_trajs)).detach()
all_embeds.shape # (N, D)

torch.Size([10000, 6])

In [84]:
normalized_all_trajs.shape

(10000, 21, 27)

In [85]:
model

TverskySIRL(
  (encoder): Sequential(
    (0): TverskyProjection(
      (feature_bank): Embedding(4, 567)
      (prototypes): Embedding(6, 567)
    )
  )
  (tversky_sim): TverskySimilarity(
    (feature_bank): Embedding(4, 6)
  )
)

In [86]:
# get feature bank of the TverskySimilarity layer used for the loss function
feature_bank = model.tversky_sim.feature_bank.weight.detach()
feature_bank.shape # 4 Tversky features, of 6-dim embeddings

torch.Size([4, 6])

In [87]:
# from tversky/test_mnist.py
# edited
import torch.nn.functional as F
def compute_salience(x, feature_bank):
    """
    x is (N, D)
    feature_bank is (F, D)
    """
    feature_measures = x @ feature_bank.T # (N, F)
    salience_measures = F.relu(feature_measures).sum(-1)
    return salience_measures

In [88]:
all_embeds = model(torch.tensor(normalized_all_trajs))
all_salience = compute_salience(all_embeds, feature_bank).detach()

In [89]:
import sys
sys.path.insert(0, '..')
import numpy as np
import plotly.graph_objects as go
from src.envs.jacorobot import Jacorobot
from src.utils.bullet_utils import waypts_to_xyz

def vis_trajectory(traj, title=""):
    env_params = {
        "object_centers": {'HUMAN_CENTER': [-0.2, -0.6, 0.9], 'LAPTOP_CENTER': [-0.5, 0.0, 0.635]},
        "resources_dir": "../data/resources",
        "horizon": 10,
        "timestep": 0.5,
        "real": False,
    }

    env = Jacorobot(
        env_params["object_centers"],
        env_params["resources_dir"],
        env_params["horizon"],
        env_params["timestep"],
        debug=False  # headless — no GUI needed
    )

    xyz = waypts_to_xyz(env.objectID["robot"], traj)

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='lines+markers'
    ))

    posH = env_params["object_centers"]["HUMAN_CENTER"]
    posL = env_params["object_centers"]["LAPTOP_CENTER"]

    fig.add_trace(go.Scatter3d(
        x=[posH[0], posL[0]],
        y=[posH[1], posL[1]],
        z=[posH[2], posL[2]],
        mode='markers',
        marker=dict(
            color=["gray", "black"],
            symbol=["cross", "square"],
            size=8,
            showscale=False
        ),
        name="human (+), laptop (square)"
    ))
    fig.update_layout(title=title)

    config = {
    'toImageButtonOptions': {
        'format': 'png', # or jpeg, svg, webp
        'scale': 4 # Increase this for higher resolution
    }
    }
    fig.show(config=config)


In [90]:

min_salience_traj = all_trajs[np.argmin(all_salience).item()]
max_salience_traj = all_trajs[np.argmax(all_salience).item()]
print(max(all_salience))

tensor(30.4391)


In [91]:
vis_trajectory(max_salience_traj, "max salience traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

In [92]:
vis_trajectory(min_salience_traj, "min salience traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

In [93]:
tversky_feature_measures = all_embeds @ feature_bank.T
tversky_feature_measures.shape

torch.Size([10000, 4])

In [94]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# to numpy
tv = tversky_feature_measures.detach().cpu().numpy()   # (10000, 4)
gt = all_features

gt_names = ["table", "laptop"]
tv_names = [f"tversky_{k}" for k in range(tv.shape[1])]

# rows = ground truth (2), cols = tversky feature (4)
fig = make_subplots(
    rows=len(gt_names), cols=tv.shape[1],
    shared_xaxes=False, shared_yaxes=False,
    horizontal_spacing=0.05, vertical_spacing=0.12,
    subplot_titles=[
        f"{g} vs {t} (r={np.corrcoef(gt[:, gi], tv[:, ti])[0,1]:.2f})"
        for gi, g in enumerate(gt_names) for ti, t in enumerate(tv_names)
    ],
)

for gi in range(len(gt_names)):
    for ti in range(tv.shape[1]):
        fig.add_trace(
            go.Scattergl(
                x=gt[:, gi], y=tv[:, ti],
                mode="markers",
                marker=dict(size=3, opacity=0.35),
                showlegend=False,
            ),
            row=gi + 1, col=ti + 1,
        )
        fig.update_xaxes(title_text=gt_names[gi], row=gi + 1, col=ti + 1)
        fig.update_yaxes(title_text=tv_names[ti], row=gi + 1, col=ti + 1)

fig.update_layout(height=600, width=1300, title="Tversky feature measures vs ground-truth features")
config = {
  'toImageButtonOptions': {
    'format': 'png', # or jpeg, svg, webp
    'scale': 4 # Increase this for higher resolution
  }
}
fig.show(config=config)

# semantic algebra?

In [95]:
# from tversky-networks-iclr2026/experiments/103-vision-nabirds/src/semantic_utils.py
def retrieve_semantic_expression(
    instance_vectors: torch.Tensor,   # (N, D)  — all instance reps from parquet
    feature_bank: torch.Tensor,        # (F, D)  — from .npy
    expression: str,
    top_feature_count: int,
    top_result_count: int,
) -> dict:
    """
    Evaluate a set expression over instance vectors and feature bank.
    Expression uses s(i) notation where i is a dataset item_id (row index).
    Example: "s(0) - s(1)"  →  features of item 0 minus features of item 1
    """
    query_item_ixes = []

    def s(item_ix: int) -> set:
        print(item_ix)
        feature_values = instance_vectors[item_ix:item_ix+1] @ feature_bank.T  # (1, F)
        print("feature_values")
        print(feature_values)
        feature_ixes = []
        for feature_ix in torch.argsort(feature_values[0], descending=True)[:top_feature_count]:
            print("loop")
            print(feature_values[0][feature_ix])
            if feature_values[0][feature_ix] > 0:
                feature_ixes.append(int(feature_ix))
            else:
                break
        query_item_ixes.append(item_ix)
        print(set(feature_ixes))
        return set(feature_ixes)

    semantic_features = eval(expression)

    if not semantic_features:
        # logging.warning("Expression evaluated to empty feature set.")
        return {"expression": expression, "query_item_ixes": [], "top_instances": [], "feature_count": 0}

    semantic_f_bank = torch.index_select(
        feature_bank, 0, torch.tensor(sorted(semantic_features))
    )
    dot = instance_vectors @ semantic_f_bank.T          # (N, |features|)
    p_saliences = F.relu(dot).sum(dim=1)                # (N,)
    p_measures  = dot.sum(dim=1)                        # (N,)

    top_instances = []
    for result_ix in torch.argsort(p_measures, descending=True)[:top_result_count]:
        top_instances.append({
            "item_ix"  : result_ix.item(),
            "salience" : p_saliences[result_ix].item(),
            "measure"  : p_measures[result_ix].item(),
        })

    return {
        "expression"      : expression.strip(),
        "query_item_ixes" : query_item_ixes,
        "feature_count"   : len(semantic_features),
        "top_instances"   : top_instances,
    }

In [96]:
table_min_traj = all_trajs[np.argmin(all_features[:,0])]
table_max_traj = all_trajs[np.argmax(all_features[:,0])]
laptop_min_traj = all_trajs[np.argmin(all_features[:,1])]
laptop_max_traj = all_trajs[np.argmax(all_features[:,1])]

In [97]:
normalized_trajs = tx(np.array([
    table_min_traj,
    table_max_traj,
    laptop_min_traj,
    laptop_max_traj
]))
normalized_trajs.shape

(4, 21, 27)

In [98]:
embeds = model(torch.tensor(normalized_trajs))

In [99]:
retrieve_semantic_expression(
    instance_vectors=embeds, #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression="s(1)-s(0)", # table_max - table_min
    top_feature_count=4,
    top_result_count=10,
)

1
feature_values
tensor([[  6.7419, -85.7574,  10.2811,   0.5759]], grad_fn=<MmBackward0>)
loop
tensor(10.2811, grad_fn=<SelectBackward0>)
loop
tensor(6.7419, grad_fn=<SelectBackward0>)
loop
tensor(0.5759, grad_fn=<SelectBackward0>)
loop
tensor(-85.7574, grad_fn=<SelectBackward0>)
{0, 2, 3}
0
feature_values
tensor([[ 3.4885, -8.9131,  0.4901,  0.8859]], grad_fn=<MmBackward0>)
loop
tensor(3.4885, grad_fn=<SelectBackward0>)
loop
tensor(0.8859, grad_fn=<SelectBackward0>)
loop
tensor(0.4901, grad_fn=<SelectBackward0>)
loop
tensor(-8.9131, grad_fn=<SelectBackward0>)
{0, 2, 3}


{'expression': 's(1)-s(0)',
 'query_item_ixes': [],
 'top_instances': [],
 'feature_count': 0}

# centering?

In [100]:
mu = all_embeds.mean(0)        # shape [6]
centered_embeds = all_embeds - mu     # [10000, 6]
centered_all_salience = compute_salience(all_embeds, feature_bank).detach()

In [101]:
example_embeds = (model(torch.tensor(normalized_trajs)) - mu).detach()

## table max - table min (high table trajs)

In [102]:
table_min_idx = np.argmin(all_features[:,0])
table_max_idx = np.argmax(all_features[:,0])
laptop_min_idx = np.argmin(all_features[:,1])
laptop_max_idx = np.argmax(all_features[:,1])

max_min_res = retrieve_semantic_expression(
    instance_vectors=centered_embeds, #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression=f"s({table_max_idx})-s({table_min_idx})", # table_max - table_min
    top_feature_count=4,
    top_result_count=10,
)

3421
feature_values
tensor([[  2.4131, -42.0225,   6.0267,   0.0709]], grad_fn=<MmBackward0>)
loop
tensor(6.0267, grad_fn=<SelectBackward0>)
loop
tensor(2.4131, grad_fn=<SelectBackward0>)
loop
tensor(0.0709, grad_fn=<SelectBackward0>)
loop
tensor(-42.0225, grad_fn=<SelectBackward0>)
{0, 2, 3}
6000
feature_values
tensor([[-0.8404, 34.8218, -3.7645,  0.3809]], grad_fn=<MmBackward0>)
loop
tensor(34.8218, grad_fn=<SelectBackward0>)
loop
tensor(0.3809, grad_fn=<SelectBackward0>)
loop
tensor(-0.8404, grad_fn=<SelectBackward0>)
{1, 3}


In [103]:
vis_trajectory(table_max_traj, "table max traj")
print("minus...")
vis_trajectory(table_min_traj, "table min traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

minus...
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaultin

In [104]:
print("equals...")

for instance in max_min_res["top_instances"]:
    idx = instance["item_ix"]
    vis_trajectory(all_trajs[idx], f"traj idx {idx}")


equals...
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulti

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

## table min - table max (should retrieve low table height trajectories)

In [105]:
table_min_idx = np.argmin(all_features[:,0])
table_max_idx = np.argmax(all_features[:,0])

min_max_res = retrieve_semantic_expression(
    instance_vectors=centered_embeds, #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression=f"s({table_min_idx})-s({table_max_idx})",
    top_feature_count=4,
    top_result_count=10,
)

6000
feature_values
tensor([[-0.8404, 34.8218, -3.7645,  0.3809]], grad_fn=<MmBackward0>)
loop
tensor(34.8218, grad_fn=<SelectBackward0>)
loop
tensor(0.3809, grad_fn=<SelectBackward0>)
loop
tensor(-0.8404, grad_fn=<SelectBackward0>)
{1, 3}
3421
feature_values
tensor([[  2.4131, -42.0225,   6.0267,   0.0709]], grad_fn=<MmBackward0>)
loop
tensor(6.0267, grad_fn=<SelectBackward0>)
loop
tensor(2.4131, grad_fn=<SelectBackward0>)
loop
tensor(0.0709, grad_fn=<SelectBackward0>)
loop
tensor(-42.0225, grad_fn=<SelectBackward0>)
{0, 2, 3}


In [106]:
print("equals...")

for instance in min_max_res["top_instances"]:
    idx = instance["item_ix"]
    vis_trajectory(all_trajs[idx], f"traj idx {idx}")


equals...
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulti

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

# analysis of mean feature values of returned expressions
uhh.. results are bit hard to tell visually. are the trajectories directional? how is the table feature even calculated??
let's compare the feature values of the two results.

In [107]:
all_features[1,0]

np.float64(0.2629035531537804)

In [108]:
max_min_table_vals = []
for instance in max_min_res["top_instances"]:
    idx = instance["item_ix"]
    max_min_table_vals.append(all_features[idx, 0])
print(np.mean(max_min_table_vals))

min_max_table_vals= []
for instance in min_max_res["top_instances"]:
    idx = instance["item_ix"]
    min_max_table_vals.append(all_features[idx, 0])
print(np.mean(min_max_table_vals))

0.7140224175242202
0.3241266846780063


In [109]:
from scipy import stats
stats.ttest_ind(max_min_table_vals, min_max_table_vals)

TtestResult(statistic=np.float64(9.646536761931905), pvalue=np.float64(1.549339663981421e-08), df=np.float64(18.0))

## laptop

In [110]:
laptop_min_idx = np.argmin(all_features[:,1])
laptop_max_idx = np.argmax(all_features[:,1])

min_max_res = retrieve_semantic_expression(
    instance_vectors=centered_embeds, #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression=f"s({laptop_min_idx})-s({laptop_max_idx})",
    top_feature_count=4,
    top_result_count=10,
)
max_min_res = retrieve_semantic_expression(
    instance_vectors=centered_embeds, #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression=f"s({laptop_max_idx})-s({laptop_min_idx})",
    top_feature_count=4,
    top_result_count=10,
)

4894
feature_values
tensor([[  9.7163, -61.4553,   5.7378,   1.8212]], grad_fn=<MmBackward0>)
loop
tensor(9.7163, grad_fn=<SelectBackward0>)
loop
tensor(5.7378, grad_fn=<SelectBackward0>)
loop
tensor(1.8212, grad_fn=<SelectBackward0>)
loop
tensor(-61.4553, grad_fn=<SelectBackward0>)
{0, 2, 3}
5018
feature_values
tensor([[ 0.9299,  3.0428, -0.6229,  0.3459]], grad_fn=<MmBackward0>)
loop
tensor(3.0428, grad_fn=<SelectBackward0>)
loop
tensor(0.9299, grad_fn=<SelectBackward0>)
loop
tensor(0.3459, grad_fn=<SelectBackward0>)
loop
tensor(-0.6229, grad_fn=<SelectBackward0>)
{0, 1, 3}
5018
feature_values
tensor([[ 0.9299,  3.0428, -0.6229,  0.3459]], grad_fn=<MmBackward0>)
loop
tensor(3.0428, grad_fn=<SelectBackward0>)
loop
tensor(0.9299, grad_fn=<SelectBackward0>)
loop
tensor(0.3459, grad_fn=<SelectBackward0>)
loop
tensor(-0.6229, grad_fn=<SelectBackward0>)
{0, 1, 3}
4894
feature_values
tensor([[  9.7163, -61.4553,   5.7378,   1.8212]], grad_fn=<MmBackward0>)
loop
tensor(9.7163, grad_fn=<Selec

In [111]:
max_min_laptop_vals = []
for instance in max_min_res["top_instances"]:
    idx = instance["item_ix"]
    max_min_laptop_vals.append(all_features[idx, 1])
print(np.mean(max_min_laptop_vals))

min_max_laptop_vals= []
for instance in min_max_res["top_instances"]:
    idx = instance["item_ix"]
    min_max_laptop_vals.append(all_features[idx, 1])
print(np.mean(min_max_laptop_vals))

0.5762385613823011
0.2774007666474871


In [112]:
from scipy import stats
stats.ttest_ind(max_min_laptop_vals, min_max_laptop_vals)

TtestResult(statistic=np.float64(7.131646278519679), pvalue=np.float64(1.2089154591254522e-06), df=np.float64(18.0))